# The gene regulatory network the model implies

Fits PWPA once on every Marson knockdown and predicts every gene the screen targeted, knocked down successfully or not.
Each prediction becomes one row of a regulator by target table, one table per cell state (`figures/grn_export.png`).
For two groups of known regulators, the similarity of their rows changes with state the way the biology does (`figures/grn_states.png`).

This fit sees every fold, the lockbox included, so it is for analysis only and is never scored.
An entry is a total predicted effect, not direct regulation, and the row of a gene that was never knocked down is only as good as its partners.

In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

sys.path.append("../utils")
import marson
from paths import data_dir

sys.path.append("../1.model")
import pwpa

torch.set_num_threads(int(os.environ.get("SLURM_CPUS_PER_TASK", os.cpu_count())))  # marson.final_model fits on first use

In [ ]:
obs, delta, readouts, symbols = marson.load()
priors = marson.priors(symbols)  # PIE's five genetic sources, looked up by each gene's symbol
best = marson.final_model(priors)  # the same fit as 0.partner_contributions.ipynb
chosen = [priors[n] for n in best["priors"]]
pd.Series(best["gates"].mean(0).numpy(), best["priors"]).round(4)

Every gene the screen targeted is a regulator, and its row is its prediction from the state's knockdowns: `G = W @ Δ`.
A knocked-down gene is never its own partner, so its row is a prediction too.
Writes, per state, the partner weights `W_<state>.npy` (regulators × the state's knockdowns, listed in `partners_<state>.tsv`) and the network `G_<state>.npy` (regulators × targets), with `regulators.tsv` and `readouts.tsv`, to `predictions/pwpa/marson_all/`.

In [ ]:
out = data_dir("predictions", "pwpa", "marson_all")
regulators = np.array(sorted(symbols))
assert set(obs.ensg) <= set(symbols)
own = pd.Index(readouts).get_indexer(regulators)  # the regulator's own readout column, -1 if it is not a readout
assert (own[pd.Index(regulators).get_indexer(obs.ensg)] == obs.target_col.values).all()
pd.DataFrame({"ensg": regulators, "symbol": [symbols[g] for g in regulators], "readout_col": own}).to_csv(
    out / "regulators.tsv", sep="\t", index=False
)
pd.DataFrame({"ensg": readouts, "symbol": [symbols.get(g, "") for g in readouts]}).to_csv(
    out / "readouts.tsv", sep="\t", index=False
)

for state in marson.STATES:
    m = (obs.condition == state).values
    partners, D = obs.ensg.values[m], delta[m]
    W = np.concatenate([
        pwpa.predict(chosen, best["scale"], best["gates"], regulators[i : i + 1024], partners, D)[1]
        for i in range(0, len(regulators), 1024)
    ])
    assert np.allclose(W.sum(1), 1, atol=1e-3)
    assert (W[pd.Index(regulators).get_indexer(partners), np.arange(len(partners))] == 0).all()  # never its own partner
    G = W @ D  # predicted expression change of every target when the regulator is knocked down
    G[np.flatnonzero(own >= 0), own[own >= 0]] = 0  # the regulator's own readout, zeroed as in the data
    obs.loc[m, ["ensg", "symbol", "fold"]].to_csv(out / f"partners_{state}.tsv", sep="\t", index=False)
    np.save(out / f"W_{state}.npy", W)
    np.save(out / f"G_{state}.npy", G)
    print(state, "regulators x partners", W.shape, "regulators x targets", G.shape)

Regulator similarity per state: for two regulators, the mean over targets of the product of their rows, after subtracting the state's mean knockdown `k`, on the 300 targets that respond most in the state.
It is high when two rows move the same targets the same way and far ("similarity × effect size").

In [ ]:
GROUPS = {
    "T-cell activation": "CD3D CD3E CD3G CD247 CD2 CD28 PTPRC LAT LCP2 ZAP70 LCK ITK VAV1 PLCG1 TMX1".split(),
    "Mediator–SAGA": "MED12 MED24 TAF6L TADA1 TADA2B SUPT20H USP22".split(),
}
genes = sum(GROUPS.values(), [])
by_symbol = {s: g for g, s in symbols.items()}
rows = pd.Index(regulators).get_indexer([by_symbol[s] for s in genes])
on = np.flatnonzero(own[rows] >= 0)

sim = {}
for state in marson.STATES:
    D = delta[(obs.condition == state).values]
    X = np.load(out / f"G_{state}.npy", mmap_mode="r")[rows] - D.mean(0)
    X[on, own[rows][on]] = 0  # the own readout is 0 in G, and stays 0 here
    X = X[:, marson.panel(D)]
    sim[state] = X @ X.T / X.shape[1]

In [ ]:
TITLE = {"Rest": "Resting", "Stim8hr": "8 h after stimulation", "Stim48hr": "48 h after stimulation"}
cmap = plt.get_cmap("PuOr").copy()  # orange: opposite, purple: alike
cmap.set_bad("0.8")
v = max(np.abs(s - np.diag(np.diag(s))).max() for s in sim.values())  # one scale for the three states

fig, axes = plt.subplots(1, 3, figsize=(20, 7.5))
for ax, (state, s) in zip(axes, sim.items()):
    im = ax.imshow(np.where(np.eye(len(s), dtype=bool), np.nan, s), cmap=cmap, vmin=-v, vmax=v)  # self pairs not drawn
    ax.set_xticks(range(len(genes)), genes, rotation=90, family="monospace", fontsize=8)
    ax.set_yticks(range(len(genes)), genes, family="monospace", fontsize=8)
    ax.xaxis.tick_top()
    ax.tick_params(length=0)
    ax.set_title(TITLE[state], fontweight="bold", pad=10)
    ax.spines[:].set_visible(False)
    start = 0
    for group in GROUPS.values():
        ax.add_patch(plt.Rectangle((start - 0.5, start - 0.5), len(group), len(group), fill=False, lw=2))
        start += len(group)
start = 0
for name, group in GROUPS.items():
    axes[-1].text(len(genes) + 0.5, start + len(group) / 2 - 0.5, name, va="center", fontweight="bold")
    start += len(group)
bar = fig.colorbar(im, ax=axes, orientation="horizontal", fraction=0.04, pad=0.04, aspect=40, ticks=[-v, 0, v])
bar.ax.set_xticklabels(["opposite", "0", "alike"])
bar.set_label("similarity × effect size")
fig.savefig("figures/grn_states.png", dpi=200, bbox_inches="tight")